# Bước 1–2: Đóng khung bài toán và Audit dữ liệu - Intel Image Classification

Notebook này phục vụ giai đoạn đầu của project PyTorch: đóng khung bài toán, kiểm tra cấu trúc dữ liệu local và thực hiện EDA ban đầu cho bộ dữ liệu Intel Image Classification.

Notebook không train model và chỉ chạy trên CPU.


## Đóng khung bài toán

- Đây là bài toán phân loại ảnh đa lớp.
- Input là ảnh cảnh tự nhiên.
- Output là nhãn lớp tương ứng với nội dung chính của ảnh.
- Notebook này tập trung vào kiểm tra dữ liệu, thống kê phân bố lớp, khảo sát kích thước ảnh và phát hiện ảnh lỗi trước khi xây dựng pipeline huấn luyện.


In [ ]:
from pathlib import Path
import os
import random
from collections import Counter

import pandas as pd
import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
    print("Seaborn ch\u01b0a \u0111\u01b0\u1ee3c c\u00e0i \u0111\u1eb7t. Notebook s\u1ebd d\u00f9ng matplotlib \u0111\u1ec3 v\u1ebd bi\u1ec3u \u0111\u1ed3.")

if sns is not None:
    sns.set_theme(style="whitegrid")
else:
    plt.style.use("default")

plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


In [ ]:
def find_project_root(start_path: Path | None = None) -> Path:
    """Tìm project root dựa trên thư mục hiện tại hoặc parent chứa thư mục data."""
    current = Path.cwd() if start_path is None else Path(start_path).resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        if (candidate / "data").exists():
            return candidate

    return current


def find_dataset_paths(project_root: Path) -> dict[str, Path]:
    """Dò train/test/pred theo path ưu tiên, sau đó thử fallback."""
    path_candidates = {
        "train": [
            project_root / "data" / "seg_train",
            project_root / "data" / "intel-image-classification" / "seg_train",
        ],
        "test": [
            project_root / "data" / "seg_test",
            project_root / "data" / "intel-image-classification" / "seg_test",
        ],
        "pred": [
            project_root / "data" / "seg_pred",
            project_root / "data" / "intel-image-classification" / "seg_pred",
        ],
    }

    resolved_paths = {}
    missing = []

    for split, candidates in path_candidates.items():
        existing_paths = [path for path in candidates if path.exists() and path.is_dir()]
        if existing_paths:
            resolved_paths[split] = existing_paths[0]
        else:
            missing.append((split, candidates))

    if missing:
        message_lines = ["Không tìm thấy đầy đủ dữ liệu Intel Image Classification.", "Các path đã thử:"]
        for split, candidates in missing:
            message_lines.append(f"- {split}:")
            for path in candidates:
                message_lines.append(f"  + {path}")
        raise FileNotFoundError("\n".join(message_lines))

    return resolved_paths


PROJECT_ROOT = find_project_root()
DATA_PATHS = find_dataset_paths(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print("Path dữ liệu đang dùng:")
for split, path in DATA_PATHS.items():
    print(f"- {split}: {path}")


In [ ]:
def list_classes(train_dir: Path) -> list[str]:
    """Lấy danh sách class từ folder con của train."""
    return sorted([path.name for path in train_dir.iterdir() if path.is_dir()])


class_names = list_classes(DATA_PATHS["train"])
test_class_names = list_classes(DATA_PATHS["test"])

print(f"Số class trong train: {len(class_names)}")
print("Danh sách class:")
for class_name in class_names:
    print(f"- {class_name}")

train_set = set(class_names)
test_set = set(test_class_names)

if train_set == test_set:
    print("\nTest có cùng danh sách class với train.")
else:
    only_train = sorted(train_set - test_set)
    only_test = sorted(test_set - train_set)
    print("\nCảnh báo: class giữa train và test chưa khớp.")
    print(f"Chỉ có trong train: {only_train if only_train else 'Không có'}")
    print(f"Chỉ có trong test: {only_test if only_test else 'Không có'}")


In [ ]:
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def get_image_files(directory: Path) -> list[Path]:
    return sorted([path for path in directory.rglob("*") if is_image_file(path)])


def count_images_by_class(split_dir: Path, split_name: str) -> pd.DataFrame:
    """Đếm số ảnh theo class cho split có cấu trúc folder class."""
    rows = []
    for class_dir in sorted([path for path in split_dir.iterdir() if path.is_dir()]):
        rows.append(
            {
                "split": split_name,
                "class_name": class_dir.name,
                "num_images": len(get_image_files(class_dir)),
            }
        )
    return pd.DataFrame(rows)


train_stats = count_images_by_class(DATA_PATHS["train"], "train")
test_stats = count_images_by_class(DATA_PATHS["test"], "test")
pred_image_files = get_image_files(DATA_PATHS["pred"])
pred_total = len(pred_image_files)

stats_df = pd.concat([train_stats, test_stats], ignore_index=True)

print("Thống kê số lượng ảnh theo lớp đã được tạo.")
print(f"Tổng ảnh pred: {pred_total}")


In [ ]:
def balance_summary(split_stats: pd.DataFrame, split_name: str, threshold: float = 0.20) -> str:
    """Nhận xét cân bằng tương đối dựa trên độ lệch max-min so với trung bình."""
    counts = split_stats["num_images"]
    if counts.empty or counts.mean() == 0:
        return f"{split_name}: chưa đủ dữ liệu để đánh giá cân bằng lớp."

    relative_range = (counts.max() - counts.min()) / counts.mean()
    if relative_range <= threshold:
        return f"{split_name}: các lớp cân bằng tương đối."
    return f"{split_name}: có chênh lệch số lượng ảnh giữa các lớp, cần lưu ý khi chia validation/training."


display(stats_df)

train_total = int(train_stats["num_images"].sum())
test_total = int(test_stats["num_images"].sum())

print(f"Tổng số ảnh train: {train_total}")
print(f"Tổng số ảnh test: {test_total}")
print(f"Tổng số ảnh pred: {pred_total}")
print(balance_summary(train_stats, "Train"))
print(balance_summary(test_stats, "Test"))


In [ ]:
def plot_class_distribution(split_stats: pd.DataFrame, split_name: str) -> None:
    plot_df = split_stats.sort_values("class_name")
    plt.figure(figsize=(10, 5))

    if sns is not None:
        ax = sns.barplot(
            data=plot_df,
            x="class_name",
            y="num_images",
            color="#4c78a8",
        )
    else:
        ax = plt.gca()
        ax.bar(plot_df["class_name"], plot_df["num_images"], color="#4c78a8")

    ax.set_title(f"Ph\u00e2n b\u1ed1 s\u1ed1 l\u01b0\u1ee3ng \u1ea3nh theo l\u1edbp - {split_name}")
    ax.set_xlabel("Class")
    ax.set_ylabel("S\u1ed1 l\u01b0\u1ee3ng \u1ea3nh")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


plot_class_distribution(train_stats, "Train")
plot_class_distribution(test_stats, "Test")


In [ ]:
def show_sample_images(train_dir: Path, class_names: list[str], samples_per_class: int = 5) -> None:
    """Hiển thị ảnh mẫu theo từng class từ train."""
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(3 * samples_per_class, 3 * n_classes))

    if n_classes == 1:
        axes = np.array([axes])

    corrupted_or_failed = []

    for row_idx, class_name in enumerate(class_names):
        image_files = get_image_files(train_dir / class_name)
        selected_files = random.sample(image_files, min(samples_per_class, len(image_files)))

        for col_idx in range(samples_per_class):
            ax = axes[row_idx, col_idx]
            ax.axis("off")

            if col_idx >= len(selected_files):
                ax.set_title(f"{class_name}\nKhông đủ ảnh")
                continue

            image_path = selected_files[col_idx]
            try:
                with Image.open(image_path) as image:
                    ax.imshow(image.convert("RGB"))
                ax.set_title(class_name)
            except Exception as exc:
                corrupted_or_failed.append((image_path, str(exc)))
                ax.set_title(f"{class_name}\nLỗi ảnh")

    plt.tight_layout()
    plt.show()

    if corrupted_or_failed:
        print("Một số ảnh mẫu không mở được:")
        for image_path, error in corrupted_or_failed[:20]:
            print(f"- {image_path}: {error}")


show_sample_images(DATA_PATHS["train"], class_names, samples_per_class=5)


In [ ]:
def inspect_image_sizes(image_files: list[Path], sample_size: int = 500) -> pd.DataFrame:
    """Kh\u1ea3o s\u00e1t k\u00edch th\u01b0\u1edbc \u1ea3nh tr\u00ean m\u1eabu ng\u1eabu nhi\u00ean."""
    if not image_files:
        return pd.DataFrame(columns=["path", "width", "height", "size"])

    selected_files = random.sample(image_files, min(sample_size, len(image_files)))
    rows = []

    for image_path in selected_files:
        try:
            with Image.open(image_path) as image:
                width, height = image.size
            rows.append(
                {
                    "path": str(image_path),
                    "width": width,
                    "height": height,
                    "size": f"{width}x{height}",
                }
            )
        except Exception as exc:
            print(f"B\u1ecf qua \u1ea3nh l\u1ed7i khi kh\u1ea3o s\u00e1t k\u00edch th\u01b0\u1edbc: {image_path} ({exc})")

    return pd.DataFrame(rows)


train_image_files = []
for class_name in class_names:
    train_image_files.extend(get_image_files(DATA_PATHS["train"] / class_name))

size_df = inspect_image_sizes(train_image_files, sample_size=500)

display(size_df.head())

if size_df.empty:
    print("Kh\u00f4ng c\u00f3 d\u1eef li\u1ec7u k\u00edch th\u01b0\u1edbc \u1ea3nh \u0111\u1ec3 th\u1ed1ng k\u00ea.")
else:
    print(f"S\u1ed1 l\u01b0\u1ee3ng k\u00edch th\u01b0\u1edbc kh\u00e1c nhau trong m\u1eabu: {size_df['size'].nunique()}")
    print("\nTop k\u00edch th\u01b0\u1edbc ph\u1ed5 bi\u1ebfn nh\u1ea5t:")
    display(size_df["size"].value_counts().head(10).rename_axis("size").reset_index(name="count"))

    print("\nMin/max width/height:")
    print(f"- Width: min={size_df['width'].min()}, max={size_df['width'].max()}")
    print(f"- Height: min={size_df['height'].min()}, max={size_df['height'].max()}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if sns is not None:
        sns.histplot(size_df["width"], bins=30, ax=axes[0], color="#4c78a8")
        sns.histplot(size_df["height"], bins=30, ax=axes[1], color="#f58518")
    else:
        axes[0].hist(size_df["width"], bins=30, color="#4c78a8", edgecolor="white")
        axes[1].hist(size_df["height"], bins=30, color="#f58518", edgecolor="white")

    axes[0].set_title("Ph\u00e2n b\u1ed1 width trong m\u1eabu train")
    axes[0].set_xlabel("Width")
    axes[0].set_ylabel("S\u1ed1 l\u01b0\u1ee3ng \u1ea3nh")

    axes[1].set_title("Ph\u00e2n b\u1ed1 height trong m\u1eabu train")
    axes[1].set_xlabel("Height")
    axes[1].set_ylabel("S\u1ed1 l\u01b0\u1ee3ng \u1ea3nh")

    plt.tight_layout()
    plt.show()


In [ ]:
def find_corrupted_images(image_files: list[Path]) -> list[tuple[Path, str]]:
    """Tìm ảnh không mở được hoặc không verify được bằng PIL."""
    corrupted = []

    for image_path in image_files:
        try:
            with Image.open(image_path) as image:
                image.verify()
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            corrupted.append((image_path, str(exc)))

    return corrupted


test_image_files = []
for class_name in test_class_names:
    test_image_files.extend(get_image_files(DATA_PATHS["test"] / class_name))

corrupted_train = find_corrupted_images(train_image_files)
corrupted_test = find_corrupted_images(test_image_files)
corrupted_all = [("train", path, error) for path, error in corrupted_train] + [
    ("test", path, error) for path, error in corrupted_test
]

print(f"Số ảnh lỗi trong train: {len(corrupted_train)}")
print(f"Số ảnh lỗi trong test: {len(corrupted_test)}")

if corrupted_all:
    print("Danh sách ảnh lỗi, tối đa 50 file đầu:")
    for split, path, error in corrupted_all[:50]:
        print(f"- [{split}] {path}: {error}")
else:
    print("Không phát hiện ảnh lỗi trong train/test.")


In [ ]:
def relative_balance_range(split_stats: pd.DataFrame) -> float | None:
    counts = split_stats["num_images"]
    if counts.empty or counts.mean() == 0:
        return None
    return float((counts.max() - counts.min()) / counts.mean())


train_balance_range = relative_balance_range(train_stats)
test_balance_range = relative_balance_range(test_stats)
size_uniform = size_df["size"].nunique() == 1 if not size_df.empty else False
num_corrupted = len(corrupted_all)

report_lines = [
    "Tổng hợp nhận xét Bước 1-2:",
    f"- Số lớp: {len(class_names)} ({', '.join(class_names)})",
    f"- Số ảnh train: {train_total}",
    f"- Số ảnh test: {test_total}",
    f"- Số ảnh pred: {pred_total}",
]

if train_balance_range is not None:
    report_lines.append(f"- Train relative range theo số ảnh/lớp: {train_balance_range:.3f}")
if test_balance_range is not None:
    report_lines.append(f"- Test relative range theo số ảnh/lớp: {test_balance_range:.3f}")

if train_balance_range is not None and train_balance_range <= 0.20:
    report_lines.append("- Phân bố lớp: cân bằng tương đối.")
else:
    report_lines.append("- Phân bố lớp: có chênh lệch giữa các lớp, nên theo dõi khi chia validation.")

if size_uniform:
    report_lines.append("- Kích thước ảnh: đồng nhất trong mẫu khảo sát.")
else:
    report_lines.append("- Kích thước ảnh: không hoàn toàn đồng nhất, cần resize/crop trong preprocessing.")

report_lines.append(f"- Ảnh lỗi train/test: {num_corrupted}")

print("\n".join(report_lines))


## Kết luận bước 1-2

Dataset đã được kiểm tra về cấu trúc thư mục, danh sách class, số lượng ảnh theo lớp, kích thước ảnh và khả năng mở ảnh bằng PIL. Nếu không có lỗi nghiêm trọng trong các cell phía trên, dữ liệu đã sẵn sàng cho bước tiếp theo.

Bước tiếp theo sẽ là tách validation từ train và xây dựng preprocessing/DataLoader cho pipeline PyTorch.
